In [2]:
import numpy as np
import pandas as pd
print(np.__version__)
print(pd.__version__)

1.26.4
2.3.3


Importing cellular automata & optimization classes, and other stuff

In [ ]:
import os
import sys
import shutil

from typing import List, Type, Callable, Dict
from numpy import int32
from numpy._typing import NDArray
import importlib

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

#from algorithm.blender import Lattice, clear_initial
from algorithm.genetic import Optimizer, Mutator, RulesetMutator, ArbitraryRulesetMutator, MutationSet
from algorithm.objectives import surface_to_vol

import numpy as np
import pandas as pd

import time

Setting up optimizer and data logging code

In [4]:
def log_mutation(data_list: List[Dict], mutations: List[MutationSet], objective_val: float):
    """
    Given the data list reference, the mutation set, and the objective value after applying it, add it to the data logging list
    """
    ic_cell_pos = []
    ic_state_old = []
    ic_state_new = []
    srt_cell_pos = []
    srt_state_old = []
    srt_state_new = []

    """
    Important difference from original genetic algorithm: 
    ic_mut is a 4 membered list showing the mutated position in the IC.
    Example: [0,0,0,1], meaning that at IC pos (0,0) the state 0 is modified to be 1.
    srt_mut is a 6 membered list showing the mutated position in the ruleset.
    Example: [0,0,0,0,0,1], meaning that at SRT rule pos (0,0,0) index 0 the rule 0 is modified to be 1.
    For a 2 state SRT there are 18 different cells for mutation: 0 - 17.
    """
    for ic_mut in mutations.ic_mutations:
        ic_cell_pos.append(tuple(ic_mut[0:2]))
        ic_state_old.append(ic_mut[2])
        ic_state_new.append(ic_mut[3])
    for srt_mut in mutations.srt_mutations:
        srt_cell_pos.append(tuple(srt_mut[0:4]))
        srt_state_old.append(srt_mut[4])
        srt_state_new.append(srt_mut[5])
    
    data_list.append({
        "ic_cell_pos": np.array(ic_cell_pos), 
        "ic_state_old": np.array(ic_state_old), 
        "ic_state_new": np.array(ic_state_new), 
        "srt_cell_pos": np.array(srt_cell_pos), 
        "srt_state_old": np.array(srt_state_old), 
        "srt_state_new": np.array(srt_state_new), 
        "objective": objective_val,
    })

def run_experiment(iters: int, grid_sz: int, ruleset_mutator_class: Type[Mutator], rule_set: List[NDArray], opt_func: Callable[[NDArray[int32]], int],
                   ic_num_mutate: int, srt_num_mutate: int, rule_mutate_prob: float, strict: bool = False, num_strict: bool = True, ic_enable: bool = True, srt_enable: bool = True):
    """
    Runs an experiment with the below hyperparameters:

    :param iters: The number of iterations the mutation algorithm (updating both IC and SRT) is going to run for
    :param grid_sz: The size of the square grid that we're going to update each iteration
    :param ruleset_mutator_class: The type of mutator used: RulesetMutator or ArbitraryRulesetMutator
    :param rule_set: The list of possible rulesets if RulesetMutator is used
    :param opt_func: The functions that gives the performance metric we're going to optimize
    :param srt_num_mutate: The number of SRT cells for which we're going to mutate the rule applied, each iteration
    :param ic_num_mutate: The number of IC cells for which we're going to mutate the rule applied, each iteration
    :param rule_mutate_prob: The probability, for each neighbor state tensor of the rule of a cell that's selected to be mutated, the final state is mutated
    :param strict: Whether SRT mutations are chosen by cell then rule or by rule directly; for more info, see mutation.py
    :param num_strict: Whether the number of SRT/IC cells mutated will be constant per iteration or variable; for more info, see mutation.py; incompatible with RulesetMutator
    :param ic_enable: Whether the IC will be mutated
    :param srt_enable: Whether the SRT will be mutated
    """
    # RESOLVED: separate SRT and IC mutations to have a certain number of each
    # RESOLVED: add a flag to enable doing only SRT or only IC mutations in an iteration (in optimizer step, and then propagate into mutator)
    ruleset_mutator = ruleset_mutator_class(rules=rule_set, grid_size=grid_sz, mutate_p=1/(grid_sz**2) * (srt_num_mutate+ic_num_mutate), rule_mutate_p=rule_mutate_prob, strict=strict, num_strict = num_strict, ic_ct = ic_num_mutate, srt_ct=srt_num_mutate, ic_enable=ic_enable, srt_enable=srt_enable)

    optim = Optimizer(mutator=ruleset_mutator, objective=lambda grid: opt_func(grid))

    """
    Pandas Dataframe used to log experiment data is:

    ic_cell_pos (np.array) | ic_state_old (np.array) | ic_state_new (np.array) | srt_cell_pos (np.array) | srt_state_old (np.array) | srt_state_new (np.array) | objective (float)
    
    etc.

    initial state for IC is in entry 0 in ic_state_old, and SRT is in entry 0 in srt_state_old

    ic and srt mutation cell positions and states can have an extra dimension in the beginning to indicate they are batch updates
    """

    init_state = optim.state
    
    data_list = [{"ic_cell_pos": grid_sz, 
                  "ic_state_old": init_state.initial, 
                  "ic_state_new": None, 
                  "srt_cell_pos": -1, 
                  "srt_state_old": init_state.rules, 
                  "srt_state_new": None, 
                  "objective": 0}]

    for it in range(iters):
        # if (it%20 == 0):
        #     print(f"Iteration {it}")
        # print(f"On iteration {it+1}...")
        accepted, new, old, mutations = optim.step()
        # data logging
        log_mutation(data_list, mutations, optim.objvalue)
        # if accepted:
        #     print("Got a better state!", optim.objvalue)

    # print(data_list)
    df = pd.DataFrame(data_list)
    # print(df)
    return df

timelogs = []

def repeat_experiment(experiment_name: str, num_expers: int, *args):
    """
    Perform (sequentially) multiple experiments that return a Pandas DataFrame and save all the data

    :param experiment_name: The name of the experiment to save the file
    :param num_expers: Number of times to run the experiment (and save all the data in one file)
    :param *args: The arguments to be passed to the experiment function
    """
    for i in range(num_expers):
        init = time.time()
        print(f'REPETITION {i}')
        ret_data = run_experiment(*args)
        timelogs.append(time.time() - init)
        print(f"Finished rep {i} in {time.time() - init}s")
        ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


Setting up experiments and gathering data

In [13]:
ITERATIONS_SET = [50, 100, 200, 500]
GRID_SIZE_SET = [10, 20, 32, 64, 100]
NUM_REPEAT = 50
EXPERIMENT_NAME = "default"

for iters in ITERATIONS_SET:
    for grid_sz in GRID_SIZE_SET:
        print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID")
        #Total param is set default to false meaning that the probability is not the total probability; 
        #so that the program aligns closer to that of the original genetic algorithm
        
        #def probability for Strict Mode = 2/3; def probability for Non-Strict Mode = 5/384
        repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, ArbitraryRulesetMutator, [[0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]], surface_to_vol, grid_sz, grid_sz**2, 2/3, True, True, True, True)

RUNNING EXPERIMENT default WITH 50 ITERATIONS AND 10 SIZE GRID
REPETITION 0
Finished rep 0 in 0.46656084060668945s
REPETITION 1


/tmp/ipykernel_7631/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


Finished rep 1 in 0.4593052864074707s
REPETITION 2


/tmp/ipykernel_7631/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


Finished rep 2 in 0.4553673267364502s
REPETITION 3


...

/tmp/ipykernel_7631/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 48
Finished rep 48 in 181.2144844532013s


/tmp/ipykernel_7631/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 49
Finished rep 49 in 180.86883997917175s


/tmp/ipykernel_7631/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


In [ ]:
#CPU MutVar Trial Timelogs
print(timelogs)

[0.46655917167663574, 0.4593038558959961, 0.4553651809692383, 0.4581420421600342, 0.4638040065765381, 0.456104040145874, 0.4599752426147461, 0.4662594795227051, 0.46102213859558105, 0.4595508575439453, 0.4587547779083252, 0.4622044563293457, 0.46880674362182617, 0.4698033332824707, 0.47233009338378906, 0.46858787536621094, 0.4675600528717041, 0.4677553176879883, 0.46842241287231445, 0.46948862075805664, 0.46241331100463867, 0.45193004608154297, 0.45564842224121094, 0.4624638557434082, 0.4602229595184326, 0.4494895935058594, 0.459902286529541, 0.4528470039367676, 0.4374735355377197, 0.44102954864501953, 0.4550471305847168, 0.4536399841308594, 0.4547770023345947, 0.4505290985107422, 0.447843074798584, 0.46430134773254395, 0.46189355850219727, 0.46035122871398926, 0.45607995986938477, 0.45786523818969727, 0.4551849365234375, 0.45935511589050293, 0.4615485668182373, 0.4594721794128418, 0.45892333984375, 0.46349024772644043, 0.4640171527862549, 0.45751070976257324, 0.4640085697174072, 0.460

In [5]:
ITERATIONS_SET = [50, 100, 200, 500]
GRID_SIZE_SET = [10, 20, 32, 64]
NUM_REPEAT = 50
EXPERIMENT_NAME = "comparative" #Directly comparable to naive implementation

for iters in ITERATIONS_SET:
    for grid_sz in GRID_SIZE_SET:
        print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID")
        #Total param is set default to false meaning that the probability is not the total probability; 
        #so that the program aligns closer to that of the original genetic algorithm
        
        #def probability for Strict Mode = 2/3; def probability for Non-Strict Mode = 5/384
        repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, ArbitraryRulesetMutator, [[0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0],[0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0]], surface_to_vol, 10, 10, 2/3, True, False, True, True)

RUNNING EXPERIMENT comparative WITH 50 ITERATIONS AND 10 SIZE GRID
REPETITION 0
Finished rep 0 in 2.031015396118164s


/tmp/ipykernel_1008909/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 1
Finished rep 1 in 0.6274557113647461s
REPETITION 2


/tmp/ipykernel_1008909/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


Finished rep 2 in 0.6244428157806396s
REPETITION 3


...

/tmp/ipykernel_1008909/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 48
Finished rep 48 in 67.36990284919739s


/tmp/ipykernel_1008909/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


REPETITION 49
Finished rep 49 in 70.057204246521s


/tmp/ipykernel_1008909/1658005238.py:116: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{experiment_name}_{i}.h5', key='data', mode='a')


In [ ]:
#CPU Naive-Comparable Trial Timelogs
print(timelogs)

[2.0310139656066895, 0.6274538040161133, 0.624441385269165, 0.5555238723754883, 0.6492831707000732, 0.6350750923156738, 0.5567312240600586, 0.6562232971191406, 0.6557910442352295, 0.6642868518829346, 0.5691947937011719, 0.6574544906616211, 0.6438090801239014, 0.6363050937652588, 0.5464866161346436, 0.6331937313079834, 0.6451315879821777, 0.6553092002868652, 0.5626654624938965, 0.661550760269165, 0.6356780529022217, 0.5960640907287598, 0.5117819309234619, 0.5706663131713867, 0.5723669528961182, 0.5375585556030273, 0.6215658187866211, 0.666893720626831, 0.5709748268127441, 0.6467900276184082, 0.6440300941467285, 0.6581165790557861, 0.559546947479248, 0.6603379249572754, 0.6507198810577393, 0.6517267227172852, 0.5584211349487305, 0.6530933380126953, 0.6367588043212891, 0.5605127811431885, 0.650421142578125, 0.6502687931060791, 0.6427586078643799, 0.6479673385620117, 0.5645232200622559, 0.6519877910614014, 0.652327299118042, 0.6490468978881836, 0.562518835067749, 0.6571049690246582, 1.8507